#Q2

In [1]:
!pip install -q -U transformers peft datasets accelerate bitsandbytes

import torch
print(torch.__version__, torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

2.14.1+cu130 True
Tesla T4
15.6 GB


In [2]:
import json, re, random
import numpy as np
from datasets import load_dataset, Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
N_TOTAL, N_TEST = 2000, 200      # 1,800 train / 200 test

In [3]:
raw = load_dataset("glaiveai/glaive-function-calling-v2", split="train")
print(raw)

def parse_example(ex):
    """Return dict(system, user, target) for examples whose FIRST assistant turn is a function call."""
    chat = ex["chat"]
    m_user = re.search(r"USER:\s*(.*?)\s*\n+\s*ASSISTANT:", chat, re.S)
    m_asst = re.search(r"ASSISTANT:\s*(.*?)\s*<\|endoftext\|>", chat, re.S)
    if not m_user or not m_asst:
        return None
    asst = m_asst.group(1).strip()
    if not asst.startswith("<functioncall>"):
        return None
    body = asst[len("<functioncall>"):].strip()

    m_name = re.search(r'"name":\s*"([^"]+)"', body)
    m_args = (re.search(r'"arguments":\s*\'(.*)\'\s*\}\s*$', body, re.S)
              or re.search(r'"arguments":\s*(\{.*\})\s*\}\s*$', body, re.S))
    if not m_name or not m_args:
        return None
    try:
        args = json.loads(m_args.group(1))
    except json.JSONDecodeError:
        return None

    system = ex["system"].replace("SYSTEM:", "", 1).strip()
    target = json.dumps({"name": m_name.group(1), "arguments": args}, ensure_ascii=False)
    return {"system": system, "user": m_user.group(1).strip(), "target": target}

# shuffle with the fixed seed, parse until we have 2,000 valid examples
idx = list(range(len(raw)))
random.Random(SEED).shuffle(idx)

parsed = []
for i in idx:
    p = parse_example(raw[i])
    if p:
        parsed.append(p)
    if len(parsed) == N_TOTAL:
        break

print(len(parsed), "examples parsed")

Dataset({
    features: ['system', 'chat'],
    num_rows: 112960
})
2000 examples parsed


In [4]:
def norm(s): return " ".join(s.lower().split())

seen, parsed = set(), []
for i in idx:
    p = parse_example(raw[i])
    if p is None:
        continue
    k = norm(p["user"])
    if k in seen:
        continue
    seen.add(k); parsed.append(p)
    if len(parsed) == N_TOTAL:
        break
print(len(parsed), "unique examples")

test_ds  = Dataset.from_list(parsed[:N_TEST])
train_ds = Dataset.from_list(parsed[N_TEST:])
tr = set(norm(e["user"]) for e in train_ds); te = set(norm(e["user"]) for e in test_ds)
print(len(train_ds), "train /", len(test_ds), "test")
print("overlap:", len(tr & te), "| dups train:", len(train_ds)-len(tr), "| dups test:", len(test_ds)-len(te))

train_ds.save_to_disk("train_ds"); test_ds.save_to_disk("test_ds")

2000 unique examples
1800 train / 200 test
overlap: 0 | dups train: 0 | dups test: 0


Saving the dataset (0/1 shards):   0%|          | 0/1800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "right"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def build_messages(ex, with_target=True):
    msgs = [{"role": "system", "content": ex["system"]},
            {"role": "user",   "content": ex["user"]}]
    if with_target:
        msgs.append({"role": "assistant", "content": ex["target"]})
    return msgs

MAX_LEN = 768

def encode(ex):
    prompt = tok.apply_chat_template(build_messages(ex, with_target=False),
                                     tokenize=False, add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    t_ids = tok(ex["target"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = (p_ids + t_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + t_ids)[:MAX_LEN]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_tok = train_ds.map(encode, remove_columns=train_ds.column_names)
print("max length:", max(len(x) for x in train_tok["input_ids"]))

def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    return {
        "input_ids":      torch.tensor([b["input_ids"] + [pad] * (m - len(b["input_ids"])) for b in batch]),
        "attention_mask": torch.tensor([b["attention_mask"] + [0] * (m - len(b["attention_mask"])) for b in batch]),
        "labels":         torch.tensor([b["labels"] + [-100] * (m - len(b["labels"])) for b in batch]),
    }

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

max length: 607


In [6]:
import gc, torch, sys
# If this still shows > 0.5 GiB, do Runtime -> Restart session and re-run cells 1-5 and 7 (leaked memory from a crashed run).
for n in ("_b","model","trainer"):
    globals().pop(n, None)
sys.last_traceback = None; sys.last_value = None; sys.last_type = None
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory in use: {torch.cuda.memory_allocated()/2**30:.2f} GiB")

GPU memory in use: 0.00 GiB


In [7]:
import time, gc, os
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, set_seed
from peft import LoraConfig, get_peft_model

TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
EPOCHS, LR, BS, GA = 3, 2e-4, 1, 16         # effective batch size 16 (BS=1 + GA=16 avoids the T4 OOM)
RESULTS_FILE = "results.json"
results = json.load(open(RESULTS_FILE)) if os.path.exists(RESULTS_FILE) else {}

def load_base():
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})

_b = load_base()
BASE_PARAMS = sum(p.numel() for p in _b.parameters())
print(f"base model parameters: {BASE_PARAMS:,}")
del _b; gc.collect(); torch.cuda.empty_cache()

from transformers import BitsAndBytesConfig

def load_4bit():
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0})

def train_run(name, peft_config, quant=False):
    set_seed(SEED)
    model = load_4bit() if quant else load_base()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()               # needed so checkpointing works with frozen embeddings
    model = get_peft_model(model, peft_config)
    for p in model.parameters():                     # fp16 AMP needs fp32 trainable weights
        if p.requires_grad:
            p.data = p.data.float()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    args = TrainingArguments(
        output_dir=f"out_{name}", num_train_epochs=EPOCHS,
        per_device_train_batch_size=BS, gradient_accumulation_steps=GA,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=17,
        weight_decay=0.0, fp16=True, optim="adamw_torch",
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=10, save_strategy="no", report_to="none",
        seed=SEED, data_seed=SEED, remove_unused_columns=False)
    trainer = Trainer(model=model, args=args, train_dataset=train_tok, data_collator=collate)

    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        out = trainer.train()
    except BaseException:
        del trainer, model; gc.collect(); torch.cuda.empty_cache()   # free GPU even if a run crashes
        raise
    minutes = (time.time() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 2**30

    model.save_pretrained(f"adapter_{name}")
    res = {"trainable_params": trainable,
           "trainable_pct": 100 * trainable / BASE_PARAMS,
           "peak_mem_gb": round(peak_gb, 2),
           "train_minutes": round(minutes, 2),
           "final_train_loss": round(out.training_loss, 4)}
    results[name] = res
    json.dump(results, open(RESULTS_FILE, "w"), indent=2)

    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return res

W1110 16:43:24.765000 13765 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 16:43:24.823000 13765 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

base model parameters: 1,543,714,304


In [8]:
lora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                      target_modules=TARGET_MODULES, task_type="CAUSAL_LM")
print(train_run("lora_r8", lora_cfg))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step,Training Loss
10,0.466403
20,0.012765
30,0.012112
40,0.006266
50,0.012138
60,0.011252
70,0.019998
80,0.010137
90,0.006218
100,0.013297


{'trainable_params': 9232384, 'trainable_pct': 0.5980629949516877, 'peak_mem_gb': 4.11, 'train_minutes': 46.38, 'final_train_loss': 0.02}


In [ ]:
qlora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                       target_modules=TARGET_MODULES, task_type="CAUSAL_LM")
print(train_run("qlora_r8", qlora_cfg, quant=True))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step,Training Loss
10,0.510272


In [ ]:
dora_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, use_dora=True,
                      target_modules=TARGET_MODULES, task_type="CAUSAL_LM")
print(train_run("dora_r8", dora_cfg))

In [ ]:
from peft import AdaLoraConfig
from transformers import TrainerCallback

# 1,800 examples / (BS*GA=16) = 113 steps per epoch (rounded up) x 3 epochs = 339 optimizer steps
TOTAL_STEPS = math.ceil(len(train_tok) / (BS * GA)) * EPOCHS if "math" in globals() else 339

class AdaLoraCallback(TrainerCallback):
    """AdaLoRA must update its rank allocation after each optimizer step, BEFORE grads are zeroed."""
    def __init__(self, peft_model): self.m = peft_model
    def on_optimizer_step(self, args, state, control, **kw):
        self.m.base_model.update_and_allocate(state.global_step)

def train_run_adalora(name, cfg):
    set_seed(SEED)
    model = load_base()
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model = get_peft_model(model, cfg)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    args = TrainingArguments(
        output_dir=f"out_{name}", num_train_epochs=EPOCHS,
        per_device_train_batch_size=BS, gradient_accumulation_steps=GA,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=17,
        weight_decay=0.0, fp16=True, optim="adamw_torch",
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=10, save_strategy="no", report_to="none",
        seed=SEED, data_seed=SEED, remove_unused_columns=False)
    trainer = Trainer(model=model, args=args, train_dataset=train_tok, data_collator=collate,
                      callbacks=[AdaLoraCallback(model)])
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        out = trainer.train()
    except BaseException:
        del trainer, model; gc.collect(); torch.cuda.empty_cache()
        raise
    minutes = (time.time() - t0) / 60
    peak_gb = torch.cuda.max_memory_allocated() / 2**30
    model.save_pretrained(f"adapter_{name}")
    res = {"trainable_params": trainable, "trainable_pct": 100 * trainable / BASE_PARAMS,
           "peak_mem_gb": round(peak_gb, 2), "train_minutes": round(minutes, 2),
           "final_train_loss": round(out.training_loss, 4)}
    results[name] = res
    json.dump(results, open(RESULTS_FILE, "w"), indent=2)
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return res

# lora_alpha=16 matches the other runs; init_r=12 -> target_r=8 (pruning happens between tinit and the last tfinal steps)
adalora_cfg = AdaLoraConfig(init_r=12, target_r=8, lora_alpha=16, lora_dropout=0.05,
                            target_modules=TARGET_MODULES, task_type="CAUSAL_LM",
                            tinit=50, tfinal=100, deltaT=10, total_step=339)
print(train_run_adalora("adalora_r12to8", adalora_cfg))

#SECTION2

In [ ]:
from peft import PeftModel

test_rows = [test_ds[i] for i in range(len(test_ds))]
gold = [json.loads(e["target"]) for e in test_rows]

@torch.no_grad()
def generate_all(model, bs=8, max_new_tokens=160):
    model.eval()
    model.config.use_cache = True
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(test_rows), bs):
        prompts = [tok.apply_chat_template(build_messages(e, with_target=False), tokenize=False,
                                           add_generation_prompt=True) for e in test_rows[i:i+bs]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = "right"
    return outs

def score(outputs):
    valid = tool_ok = full = 0
    for out, g in zip(outputs, gold):
        try:
            p = json.loads(out.strip())          # strict: extra text before/after counts as INVALID
        except Exception:
            continue
        valid += 1
        if isinstance(p, dict) and p.get("name") == g["name"]:
            tool_ok += 1
            if p.get("arguments") == g["arguments"]:
                full += 1
    n = len(outputs)
    return {"json_valid": valid, "json_valid_pct": 100*valid/n,
            "tool_acc": tool_ok, "tool_acc_pct": 100*tool_ok/n,
            "full_match": full, "full_match_pct": 100*full/n}

def evaluate(name, quant=False, adapter=None):
    model = load_4bit() if quant else load_base()
    if adapter:
        model = PeftModel.from_pretrained(model, adapter)
    outs = generate_all(model)
    json.dump(outs, open(f"preds_{name}.json", "w"))
    sc = score(outs)
    results.setdefault(name, {}).update(sc)
    json.dump(results, open(RESULTS_FILE, "w"), indent=2)
    del model; gc.collect(); torch.cuda.empty_cache()
    return sc

# Base model: no adapter, no training (the "before" row)
results.setdefault("base", {}).update({"trainable_params": 0, "trainable_pct": 0.0,
                                       "peak_mem_gb": None, "train_minutes": 0.0})
print("base       ", evaluate("base"))

In [ ]:
# Evaluate every trained adapter on the same 200 held-out examples
print("lora_r8    ", evaluate("lora_r8",  adapter="adapter_lora_r8"))
print("qlora_r8   ", evaluate("qlora_r8", quant=True, adapter="adapter_qlora_r8"))
print("dora_r8    ", evaluate("dora_r8",  adapter="adapter_dora_r8"))
print("adalora    ", evaluate("adalora_r12to8", adapter="adapter_adalora_r12to8"))

In [ ]:
import pandas as pd
order = ["base", "lora_r8", "qlora_r8", "dora_r8", "adalora_r12to8"]
cols = ["trainable_params", "trainable_pct", "peak_mem_gb", "train_minutes",
        "json_valid", "tool_acc", "full_match"]
df = pd.DataFrame({k: results[k] for k in order if k in results}).T.reindex(columns=cols)
df.columns = ["Trainable params", "Trainable %", "Peak GPU mem (GiB)", "Train time (min)",
              "JSON valid /200", "Tool acc /200", "Full match /200"]
df["Tool acc %"] = df["Tool acc /200"] / 2
df["Full match %"] = df["Full match /200"] / 2
display(df)
df.to_csv("section2_results.csv")

#SECTION3

In [ ]:

cands = ["lora_r8", "qlora_r8", "dora_r8", "adalora_r12to8"]
ranked = sorted([c for c in cands if c in results and "tool_acc" in results[c]],
                key=lambda c: (-results[c]["tool_acc"], -results[c]["full_match"], results[c]["peak_mem_gb"]))
BEST = ranked[0]
print("ranking:", ranked)
print("Winner:", BEST)

METHOD = {"lora_r8": "lora", "qlora_r8": "qlora", "dora_r8": "dora", "adalora_r12to8": "adalora"}[BEST]

def make_cfg(method, r):
    """Only the rank changes. lora_alpha = 2*r keeps alpha/r = 2, the same scaling as the r=8, alpha=16 runs."""
    if method == "adalora":
        return AdaLoraConfig(init_r=int(1.5*r), target_r=r, lora_alpha=2*r, lora_dropout=0.05,
                             target_modules=TARGET_MODULES, task_type="CAUSAL_LM",
                             tinit=50, tfinal=100, deltaT=10, total_step=339)
    return LoraConfig(r=r, lora_alpha=2*r, lora_dropout=0.05, target_modules=TARGET_MODULES,
                      task_type="CAUSAL_LM", use_dora=(method == "dora"))

In [ ]:
RANKS = [4, 16, 32]            # r=8 already trained above
for r in RANKS:
    name = f"{METHOD}_r{r}"
    if name in results and "tool_acc" in results[name]:
        print("skip (already done):", name); continue
    if name not in results:
        cfg = make_cfg(METHOD, r)
        if METHOD == "adalora":
            print(train_run_adalora(name, cfg))
        else:
            print(train_run(name, cfg, quant=(METHOD == "qlora")))
    print(name, evaluate(name, quant=(METHOD == "qlora"), adapter=f"adapter_{name}"))

In [ ]:
import matplotlib.pyplot as plt

rank_keys = {4: f"{METHOD}_r4", 8: BEST, 16: f"{METHOD}_r16", 32: f"{METHOD}_r32"}
rows = [(r, results[k]) for r, k in rank_keys.items() if k in results and "tool_acc" in results[k]]
rs      = [r for r, _ in rows]
acc     = [d["tool_acc_pct"] for _, d in rows]
full    = [d["full_match_pct"] for _, d in rows]
mem     = [d["peak_mem_gb"] for _, d in rows]
params  = [d["trainable_params"] for _, d in rows]
mins    = [d["train_minutes"] for _, d in rows]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(rs, acc, "o-", label="Tool accuracy")
ax.plot(rs, full, "s--", label="Full match")
ax.set_xscale("log", base=2); ax.set_xticks(rs); ax.set_xticklabels(rs)
ax.set_xlabel("LoRA rank r"); ax.set_ylabel("% of 200 test examples")
ax.set_title(f"Accuracy vs rank ({METHOD.upper()})"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig("rank_vs_accuracy.png", dpi=150); plt.show()

# 95% Wilson interval on tool accuracy: with n=200 differences of 1-3 points are within noise
def wilson(k, n=200, z=1.96):
    p = k / n; c = (p + z*z/(2*n)) / (1 + z*z/n)
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return 100*(c-h), 100*(c+h)

tab = pd.DataFrame({"rank": rs, "trainable params": params, "peak mem (GiB)": mem, "train min": mins,
                    "tool acc %": acc, "full match %": full,
                    "tool acc 95% CI": [f"{wilson(d['tool_acc'])[0]:.1f}-{wilson(d['tool_acc'])[1]:.1f}" for _, d in rows]})
display(tab)
tab.to_csv("section3_rank_sweep.csv", index=False)

#SECTION4

In [ ]:
# ---- Section 4, item 6: where the best model still breaks ----
preds = json.load(open(f"preds_{BEST}.json"))
train_names = {json.loads(e["target"])["name"] for e in train_ds}

def classify(out, g):
    s = out.strip()
    try:
        p = json.loads(s)
    except Exception:
        return "refused to call a tool" if "{" not in s else "broken JSON"
    if not isinstance(p, dict) or "name" not in p:
        return "other (wrong schema)"
    if p["name"] != g["name"]:
        return "wrong tool" if p["name"] in train_names else "made-up function name"
    if p.get("arguments") == g["arguments"]:
        return None                                   # correct
    pa, ga = p.get("arguments", {}), g["arguments"]
    if not isinstance(pa, dict):
        return "other (arguments not an object)"
    if set(pa) - set(ga):
        return "made-up argument"
    if set(ga) - set(pa):
        return "missing argument"
    return "wrong argument value"

errors = []
for e, out, g in zip(test_rows, preds, gold):
    label = classify(out, g)
    if label:
        errors.append({"user": e["user"], "gold": e["target"], "model": out.strip(), "label": label})

print(f"{BEST}: {len(errors)} wrong out of {len(preds)}")
print(pd.Series([x["label"] for x in errors]).value_counts(), "\n")

# pick 5 examples, spreading across error types first, then filling up
chosen, seen_labels = [], set()
for x in errors:
    if x["label"] not in seen_labels:
        chosen.append(x); seen_labels.add(x["label"])
    if len(chosen) == 5: break
for x in errors:
    if len(chosen) == 5: break
    if x not in chosen: chosen.append(x)

for i, x in enumerate(chosen, 1):
    print(f"--- Failure {i}: {x['label']}")
    print("Customer :", x["user"][:300])
    print("Correct  :", x["gold"][:300])
    print("Model    :", x["model"][:300], "\n")
json.dump(chosen, open("five_failures.json", "w"), indent=2)

In [ ]:
# ---- Section 4, item 5: inference speed, merged vs unmerged ----
def bench(label, model, n=48, bs=8, max_new_tokens=128):
    model.eval(); model.config.use_cache = True
    tok.padding_side = "left"
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    new_tokens, t0 = 0, time.time()
    with torch.no_grad():
        for i in range(0, n, bs):
            prompts = [tok.apply_chat_template(build_messages(e, with_target=False), tokenize=False,
                       add_generation_prompt=True) for e in test_rows[i:i+bs]]
            enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
            new_tokens += int((gen[:, enc["input_ids"].shape[1]:] != tok.pad_token_id).sum())
    torch.cuda.synchronize(); dt = time.time() - t0
    tok.padding_side = "right"
    return {"setup": label, "tokens/sec": round(new_tokens/dt, 1), "sec per example": round(dt/n, 3),
            "peak mem GiB": round(torch.cuda.max_memory_allocated()/2**30, 2)}

rows = []
for name, quant in [("lora_r8", False), ("dora_r8", False), ("qlora_r8", True)]:
    if name not in results: continue
    m = PeftModel.from_pretrained(load_4bit() if quant else load_base(), f"adapter_{name}")
    rows.append(bench(f"{name} adapter (unmerged)", m))
    if not quant:                                      # merging into 4-bit weights is lossy, so QLoRA stays unmerged
        m = m.merge_and_unload()
        rows.append(bench(f"{name} merged", m))
    del m; gc.collect(); torch.cuda.empty_cache()
display(pd.DataFrame(rows))

#Q4


In [4]:
import math
import numpy as np
import pandas as pd
from collections import Counter

pd.set_option("display.precision", 4)
LABELS = ["Positive", "Negative", "Neutral"]

# Rows = actual, columns = predicted
CM_STAGE1 = np.array([[100,  5, 15],
                      [  8, 80, 12],
                      [ 20, 10, 50]])
CM_STAGE2 = np.array([[106,  4, 10],
                      [  6, 84, 10],
                      [  9,  7, 64]])
Z = 1.96  # 95% two-sided

def per_class_metrics(cm):
    cm = np.asarray(cm, dtype=float)
    tp = np.diag(cm)
    row = cm.sum(axis=1)   # actual totals (support)
    col = cm.sum(axis=0)   # predicted totals
    precision = tp / col
    recall = tp / row
    f1 = 2 * precision * recall / (precision + recall)
    return pd.DataFrame({"TP": tp.astype(int), "Support": row.astype(int),
                         "Predicted": col.astype(int),
                         "Precision": precision, "Recall": recall, "F1": f1},
                        index=LABELS)

def averaged_f1(cm):
    cm = np.asarray(cm, dtype=float)
    df = per_class_metrics(cm)
    n = cm.sum()
    macro = df["F1"].mean()
    weighted = (df["F1"] * df["Support"]).sum() / n
    micro = np.diag(cm).sum() / n          # micro-P = micro-R = accuracy
    return macro, weighted, micro

def accuracy_ci(cm, z=Z):
    cm = np.asarray(cm)
    n = cm.sum()
    p = np.diag(cm).sum() / n
    se = math.sqrt(p * (1 - p) / n)
    wald = (p - z * se, p + z * se)
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    wilson = (centre - half, centre + half)
    return p, se, wald, wilson

def full_report(cm, name):
    print(f"===== {name} =====")
    print(cm_to_df(cm), "\n")
    p, se, wald, wilson = accuracy_ci(cm)
    print(f"Accuracy = {int(np.diag(cm).sum())}/{int(np.sum(cm))} = {p:.4f}")
    print(f"Standard error = {se:.5f}")
    print(f"95% CI (Wald)   = [{wald[0]:.4f}, {wald[1]:.4f}]")
    print(f"95% CI (Wilson) = [{wilson[0]:.4f}, {wilson[1]:.4f}]\n")
    print(per_class_metrics(cm), "\n")
    macro, weighted, micro = averaged_f1(cm)
    print(f"Macro F1    = {macro:.4f}")
    print(f"Weighted F1 = {weighted:.4f}")
    print(f"Micro F1    = {micro:.4f}  (equals accuracy)\n")

def cm_to_df(cm):
    df = pd.DataFrame(cm, index=[f"Actual {l}" for l in LABELS],
                      columns=[f"Pred {l}" for l in LABELS])
    df["Row total"] = df.sum(axis=1)
    df.loc["Column total"] = df.sum()
    return df

full_report(CM_STAGE1, 'Stage 1: zero-shot')

errs = []
for i, a in enumerate(LABELS):
    for j, p in enumerate(LABELS):
        if i != j:
            errs.append((f"{a} -> {p}", int(CM_STAGE1[i, j])))
errs.sort(key=lambda x: -x[1])
print(pd.DataFrame(errs, columns=["Error (actual -> predicted)", "Count"]).to_string(index=False))

bal = pd.DataFrame({"Actual": CM_STAGE1.sum(1), "Predicted": CM_STAGE1.sum(0)}, index=LABELS)
print("\n", bal)
print("\nObservations: Neutral has the lowest F1; the largest error is Neutral -> Positive; "
      "Positive is over-predicted (128 vs 120).")

full_report(CM_STAGE2, 'Stage 2: improved prompt')

m1, w1, mi1 = averaged_f1(CM_STAGE1)
m2, w2, mi2 = averaged_f1(CM_STAGE2)
pc1, pc2 = per_class_metrics(CM_STAGE1), per_class_metrics(CM_STAGE2)

comp = pd.DataFrame({
    "Stage 1": [mi1, m1, w1, pc1.loc["Neutral","F1"], pc1.loc["Neutral","Recall"], pc1.loc["Neutral","Precision"]],
    "Stage 2": [mi2, m2, w2, pc2.loc["Neutral","F1"], pc2.loc["Neutral","Recall"], pc2.loc["Neutral","Precision"]],
}, index=["Accuracy / micro-F1", "Macro F1", "Weighted F1", "Neutral F1", "Neutral recall", "Neutral precision"])
comp["Change"] = comp["Stage 2"] - comp["Stage 1"]
print(comp)

print("\nPer-class F1 change:")
print((pc2["F1"] - pc1["F1"]).round(4))

# Do the 95% CIs overlap?
_, _, w_1, _ = accuracy_ci(CM_STAGE1)
_, _, w_2, _ = accuracy_ci(CM_STAGE2)
print(f"\nStage 1 CI = [{w_1[0]:.3f}, {w_1[1]:.3f}], Stage 2 CI = [{w_2[0]:.3f}, {w_2[1]:.3f}]")
print("CIs overlap:", w_1[1] > w_2[0], "-> a paired test (McNemar) on per-review predictions would be more powerful.")

runs = np.array([0.847, 0.840, 0.853, 0.837, 0.850])
mean = runs.mean()
sd_sample = runs.std(ddof=1)
sd_pop = runs.std(ddof=0)
t_crit = 2.776  # t(0.975, df=4)
half = t_crit * sd_sample / math.sqrt(len(runs))

print(f"Mean = {mean:.4f}")
print(f"Sample SD (n-1) = {sd_sample:.4f}   Population SD (n) = {sd_pop:.4f}")
print(f"Report: {mean:.3f} +/- {sd_sample:.3f}")
print(f"Range = {runs.min():.3f} to {runs.max():.3f}  (spread {100*(runs.max()-runs.min()):.1f} points)")
print(f"Coefficient of variation = {100*sd_sample/mean:.2f}%")
print(f"95% CI for the mean (t, df=4) = [{mean-half:.4f}, {mean+half:.4f}]")

_, se_test, _, _ = accuracy_ci(CM_STAGE2)
print(f"\nTest-set sampling SE (n=300) = {se_test:.4f}  vs  ordering SD = {sd_sample:.4f}")
print("Ordering noise is about a third of the sampling noise -> the prompt is stable.")

reference = "the delivery was late but the product quality is good".split()
candidate = "the product quality is good but delivery was slow".split()
print("Reference :", len(reference), "tokens")
print("Candidate :", len(candidate), "tokens")

def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

# ---------- BLEU-4 ----------
precisions = []
for n in range(1, 5):
    c, r = ngrams(candidate, n), ngrams(reference, n)
    overlap = sum((c & r).values())       # clipped counts
    total = sum(c.values())
    precisions.append(overlap / total)
    print(f"  p{n} = {overlap}/{total} = {overlap/total:.4f}")

c_len, r_len = len(candidate), len(reference)
bp = 1.0 if c_len > r_len else math.exp(1 - r_len / c_len)
geo = math.exp(sum(math.log(p) for p in precisions) / 4)
bleu4 = bp * geo
print(f"Geometric mean = {geo:.4f}, brevity penalty = {bp:.4f}")
print(f"BLEU-4 = {bleu4:.4f}\n")

# ---------- ROUGE-N ----------
def rouge_n(n):
    c, r = ngrams(candidate, n), ngrams(reference, n)
    ov = sum((c & r).values())
    p, rec = ov / sum(c.values()), ov / sum(r.values())
    f = 2 * p * rec / (p + rec) if ov else 0.0
    return ov, p, rec, f

for n in (1, 2):
    ov, p, r, f = rouge_n(n)
    print(f"ROUGE-{n}: overlap={ov}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")

# ---------- ROUGE-L (LCS) ----------
def lcs(a, b):
    L = [[0]*(len(b)+1) for _ in range(len(a)+1)]
    for i in range(len(a)):
        for j in range(len(b)):
            L[i+1][j+1] = L[i][j] + 1 if a[i] == b[j] else max(L[i][j+1], L[i+1][j])
    return L[-1][-1]

l = lcs(candidate, reference)
p_l, r_l = l / len(candidate), l / len(reference)
f_l = 2 * p_l * r_l / (p_l + r_l)
print(f"ROUGE-L: LCS={l}  P={p_l:.4f}  R={r_l:.4f}  F1={f_l:.4f}\n")

# ---------- METEOR (exact match) ----------
def meteor_exact(cand, ref, alpha=0.9, beta=3.0, gamma=0.5):
    # Greedy exact alignment that picks, for each candidate token, the reference
    # position that extends the current chunk if possible (minimises chunks).
    used, align = set(), []
    for i, w in enumerate(cand):
        options = [j for j, x in enumerate(ref) if x == w and j not in used]
        if not options:
            continue
        prev = align[-1][1] if align else None
        pick = next((j for j in options if prev is not None and j == prev + 1), None)
        if pick is None:
            pick = options[0]
        used.add(pick)
        align.append((i, pick))
    m = len(align)
    if m == 0:
        return 0.0, 0, 0
    # count chunks: contiguous in BOTH candidate and reference
    chunks = 1
    for (i1, j1), (i2, j2) in zip(align, align[1:]):
        if not (i2 == i1 + 1 and j2 == j1 + 1):
            chunks += 1
    P, R = m / len(cand), m / len(ref)
    fmean = P * R / (alpha * P + (1 - alpha) * R)
    penalty = gamma * (chunks / m) ** beta
    return fmean * (1 - penalty), m, chunks

# The greedy pass above can pick a worse alignment for repeated words ("the" appears twice
# in the reference), so we also brute-force the best alignment of "the".
import itertools
best = None
the_ref_positions = [j for j, x in enumerate(reference) if x == "the"]
for the_pos in the_ref_positions:
    ref_mod = list(reference)
    for j in the_ref_positions:
        if j != the_pos:
            ref_mod[j] = f"__{j}__"      # hide the other "the"
    score, m, chunks = meteor_exact(candidate, ref_mod)
    if best is None or score > best[0]:
        best = (score, m, chunks, the_pos)

score, m, chunks, the_pos = best
P, R = m / len(candidate), m / len(reference)
fmean = 10 * P * R / (R + 9 * P)
penalty = 0.5 * (chunks / m) ** 3
print(f"METEOR: matches={m}, chunks={chunks}, P={P:.4f}, R={R:.4f}, Fmean={fmean:.4f}, penalty={penalty:.4f}")
print(f"METEOR = {fmean*(1-penalty):.4f}")

summary = pd.Series({"BLEU-4": bleu4, "ROUGE-1 F1": rouge_n(1)[3], "ROUGE-2 F1": rouge_n(2)[3],
                     "ROUGE-L F1": f_l, "METEOR": fmean*(1-penalty)})
print("\nSummary:\n", summary.round(4))

token_probs = np.array([0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20])
cand_tokens = candidate  # 9 generated tokens, in order

nll = -np.log(token_probs)
tbl = pd.DataFrame({"token": cand_tokens, "p": token_probs, "-ln p": nll})
print(tbl.to_string(index=False))

avg_nll = nll.mean()
ppl = math.exp(avg_nll)
print(f"\nSum of -ln p = {nll.sum():.4f}")
print(f"Average NLL  = {avg_nll:.4f}")
print(f"Perplexity   = exp({avg_nll:.4f}) = {ppl:.3f}")
print("\nInterpretation: the model behaved as if choosing among ~2.3 equally likely tokens at each "
      "step, i.e. fairly confident. The least certain tokens were 'but', 'delivery' and 'slow'. "
      "Low perplexity measures confidence in its own output, not correctness vs the reference.")

===== Stage 1: zero-shot =====
                 Pred Positive  Pred Negative  Pred Neutral  Row total
Actual Positive            100              5            15        120
Actual Negative              8             80            12        100
Actual Neutral              20             10            50         80
Column total               128             95            77        300 

Accuracy = 230/300 = 0.7667
Standard error = 0.02442
95% CI (Wald)   = [0.7188, 0.8145]
95% CI (Wilson) = [0.7156, 0.8110]

           TP  Support  Predicted  Precision  Recall      F1
Positive  100      120        128     0.7812  0.8333  0.8065
Negative   80      100         95     0.8421  0.8000  0.8205
Neutral    50       80         77     0.6494  0.6250  0.6369 

Macro F1    = 0.7546
Weighted F1 = 0.7659
Micro F1    = 0.7667  (equals accuracy)

Error (actual -> predicted)  Count
        Neutral -> Positive     20
        Positive -> Neutral     15
        Negative -> Neutral     12
        Neutral -> 